# 01 – Data Preparation

Loads the SPARROW HUC-12 input table and TREND-N nitrogen surplus data,
computes N budget variables, remaps network node IDs, calculates the
hydrologic sequencing (HYDSEQ) needed for upstream-to-downstream routing,
and assembles per-year PyTorch DataLoaders used by the training and
validation notebooks.


In [ ]:
import os, warnings, pickle
import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader, TensorDataset
from sklearn.preprocessing import MinMaxScaler

warnings.filterwarnings('ignore')

from sparrow.utils import hydseq, setup_seed

# ── Paths (edit these to point to your data) ─────────────────────────────────
# sparrow_input_path : one row per HUC-12 reach × year (see column list below)
# trendn_path        : annual N surplus from TREND-N (see column list below)
# cv_fold_dir        : folder containing CV_1_data.csv … CV_5_data.csv (notebook 03)
# output_dir         : where prepared data will be saved

sparrow_input_path = 'data/sparrow_input.csv'
trendn_path        = 'data/trendn_surplus.csv'
cv_fold_dir        = 'data/5_fold_spatial/'
output_dir         = 'outputs/prepared/'
os.makedirs(output_dir, exist_ok=True)


## 1. Load input data

### Expected columns in `sparrow_input.csv`

| Column | Type | Description |
|---|---|---|
| `waterid` | int | Unique reach ID (HUC-12 scale, with 2-digit year suffix) |
| `Year` | int | Calendar year |
| `fnode`, `tnode` | int | From-node and to-node IDs defining stream network connectivity |
| `rchtype` | int | Reach type flag |
| `headflag` | int | 1 = headwater reach |
| `frac` | float | Fraction of reach load routed downstream |
| `iftran` | int | 1 = in-transit reach |
| `depvar` | float | Observed annual TN load at gauging station (kg/yr); 0 or NaN = ungauged |
| `strmloss` | float | In-stream reach decay variable |
| `iresload` | float | Reservoir load variable |
| `demiarea` | float | Catchment area (km²; converted to ha internally) |
| `slope` | float | Reach slope |
| `meanq` | float | Mean annual discharge |
| `PPT` | float | Annual total precipitation |
| `meanTemp` | float | Mean annual air temperature |
| `tiles_perc` | float | Tile drainage fraction |
| `soil_CLAYAVE` | float | Mean soil clay content |
| `CRP_percent` | float | Conservation Reserve Program area fraction |
| `no_till` | float | No-till area fraction in agricultural land |
| `cover_crop_percent` | float | Cover crop area fraction |
| `forest_percent` | float | Forest area fraction |
| `wetlands_percent` | float | Wetlands area fraction |

### Expected columns in `trendn_surplus.csv`

| Column | Type | Description |
|---|---|---|
| `waterid` | int | Reach ID (matches `sparrow_input.csv`) |
| `Year` | int | Calendar year |
| `NSurplus` | float | Annual N surplus from TREND-N (kg N ha⁻¹ yr⁻¹) |
| `Atmospheric_Oxidized` | float | Oxidized atmospheric N deposition (kg N ha⁻¹ yr⁻¹) |
| `Atmospheric_Reduced` | float | Reduced atmospheric N deposition (kg N ha⁻¹ yr⁻¹) |
| `Fertilizer_Agriculture` | float | Agricultural synthetic fertiliser N (kg N ha⁻¹ yr⁻¹) |
| `Fertilizer_NonAgriculture` | float | Non-agricultural synthetic fertiliser N (kg N ha⁻¹ yr⁻¹) |
| `Fix_Cropland` | float | Biological N fixation on cropland (kg N ha⁻¹ yr⁻¹) |
| `Fix_Pasture` | float | Biological N fixation on pasture (kg N ha⁻¹ yr⁻¹) |
| `Human` | float | Human wastewater N (kg N ha⁻¹ yr⁻¹) |
| `Lvst_BeefCow` … `Lvst_Turkeys` | float | Livestock manure N by species (kg N ha⁻¹ yr⁻¹) |
| `CropUptake_Cropland` | float | Crop N uptake on cropland (kg N ha⁻¹ yr⁻¹) |
| `CropUptake_Pasture` | float | Crop N uptake on pasture (kg N ha⁻¹ yr⁻¹) |


In [ ]:
input_data_huc12 = pd.read_csv(sparrow_input_path)
input_data_huc12['deliver_0'] = 0   # placeholder delivery column; delivery factor = exp(0) = 1

surplus = pd.read_csv(trendn_path)
surplus['waterid'] = surplus['waterid'].astype(int)
input_data_huc12 = input_data_huc12.merge(surplus, on='waterid', how='left')
input_data_huc12 = input_data_huc12.fillna(0)

print(f"Loaded {len(input_data_huc12):,} reach-year rows, "
      f"{input_data_huc12['waterid'].nunique():,} unique reaches, "
      f"{input_data_huc12['Year'].nunique()} years")


## 2. Compute N budget variables

Total N surplus (kg ha⁻¹ yr⁻¹) is multiplied by catchment area to give the
single SPARROW source term `total_N_surplus` (kg yr⁻¹).  Individual
N-budget components are also converted to kg yr⁻¹ for use in scenario
attribution (notebook 05).


In [ ]:
input_data_huc12['area_ha'] = input_data_huc12['demiarea'] * 100   # km² → ha

cols_inputs  = ['Atmospheric_Oxidized', 'Atmospheric_Reduced',
                'Fertilizer_Agriculture', 'Fertilizer_NonAgriculture',
                'Fix_Cropland', 'Fix_Pasture', 'Human',
                'Lvst_BeefCow', 'Lvst_Broilers', 'Lvst_DairyCow', 'Lvst_Equine',
                'Lvst_Hogs', 'Lvst_LayersPullets', 'Lvst_OtherCattle',
                'Lvst_SheepGoat', 'Lvst_Turkeys']
cols_uptakes = ['CropUptake_Cropland', 'CropUptake_Pasture']

input_data_huc12['Total_N_Input'] = input_data_huc12[cols_inputs].sum(axis=1)
input_data_huc12['Total_Uptake']  = input_data_huc12[cols_uptakes].sum(axis=1)
input_data_huc12['N_surplus']     = (input_data_huc12['Total_N_Input']
                                     - input_data_huc12['Total_Uptake'])

# Convert all per-ha columns to total kg/yr
cols_to_convert = cols_inputs + cols_uptakes + ['NSurplus', 'N_surplus']
for col in cols_to_convert:
    if col in input_data_huc12.columns:
        input_data_huc12[f'total_{col}'] = input_data_huc12[col] * input_data_huc12['area_ha']

# SPARROW source: total N surplus (kg/yr)
input_data_huc12['total_N_surplus'] = input_data_huc12['N_surplus'] * input_data_huc12['area_ha']

# Aggregated component groups (used in notebook 05)
input_data_huc12['FARM_N_surplus'] = (
    input_data_huc12['total_Fertilizer_Agriculture'] +
    input_data_huc12['total_Fertilizer_NonAgriculture'] -
    input_data_huc12['total_CropUptake_Cropland'] -
    input_data_huc12['total_CropUptake_Pasture'])
input_data_huc12['Fix_surplus']   = (input_data_huc12['total_Fix_Cropland'] +
                                      input_data_huc12['total_Fix_Pasture'])
input_data_huc12['ndep_surplus']  = (input_data_huc12['total_Atmospheric_Oxidized'] +
                                      input_data_huc12['total_Atmospheric_Reduced'])
print("N budget variables computed.")


## 3. Define model column groups

In [ ]:
source_columns    = ['total_N_surplus']   # single N source
delivery_columns  = ['deliver_0']         # constant zero → delivery factor = 1
stream_columns    = ['strmloss']
reservoir_columns = ['iresload']

# ParamGenerator input features
input_columns = [
    'PPT',           # annual total precipitation
    'tiles_perc',          # tile drainage fraction
    'soil_CLAYAVE',        # mean soil clay content
    'meanTemp',            # mean annual air temperature
    'CRP_percent',         # Conservation Reserve Program area fraction
    'no_till',             # no-till area fraction in agricultural land
    'cover_crop_percent',  # cover crop area fraction
    'forest_percent',      # forest area fraction
    'wetlands_percent',    # wetlands area fraction
]
input_columns_strm = ['slope', 'meanq']    # reach slope, mean annual discharge
input_columns_res  = ['meanTemp']          # mean annual air temperature

print(f"Catchment features ({len(input_columns)}): {input_columns}")
print(f"Stream features    ({len(input_columns_strm)}): {input_columns_strm}")
print(f"Reservoir features ({len(input_columns_res)}): {input_columns_res}")


## 4. Remap node IDs and compute hydrologic sequence (HYDSEQ)

In [ ]:
# Sequential reach index (1…N) for compact GPU tensor indexing
input_data_huc12['new_waterid'] = range(1, len(input_data_huc12) + 1)

all_nodes    = pd.unique(input_data_huc12[['fnode', 'tnode']].values.ravel('K'))
node_mapping = {n: i + 1 for i, n in enumerate(all_nodes) if pd.notna(n)}

input_data_huc12['from_node'] = input_data_huc12['fnode'].map(node_mapping)
input_data_huc12['to_node']   = input_data_huc12['tnode'].map(node_mapping)

nan_mask = input_data_huc12['to_node'].isna()
start_id = len(input_data_huc12) + 2
input_data_huc12.loc[nan_mask, 'to_node'] = range(start_id, start_id + nan_mask.sum())

# HYDSEQ: negative integer that sorts reaches upstream → downstream
df_hseq = hydseq(input_data_huc12[['waterid', 'fnode', 'tnode']])
if 'hydseq' in input_data_huc12.columns:
    input_data_huc12 = input_data_huc12.drop(columns=['hydseq'])
input_data_huc12 = input_data_huc12.merge(
    df_hseq[['seqvar', 'hydseq']], left_index=True, right_on='seqvar', how='left')
print("Node remapping and HYDSEQ done.")


## 5. Build design matrix and mean-centre delivery variables

`deliver_0` is all zeros, so the design matrix is a 1×1 zero matrix and the
delivery factor `exp(Z_D × θ_D × dlvdsgn)` evaluates to 1 for every reach.
The mean-centering step follows standard SPARROW convention.


In [ ]:
# Reorder columns: sources first, then delivery, then everything else
other_cols = [c for c in input_data_huc12.columns
              if c not in source_columns + delivery_columns]
input_data_huc12 = input_data_huc12[source_columns + delivery_columns + other_cols]

# Design matrix: [n_sources × n_delivery_vars] — all zeros here
dlvdsgn_array = np.zeros((len(source_columns), len(delivery_columns)), dtype=np.float32)
dlvdsgn       = torch.tensor(dlvdsgn_array, dtype=torch.float32)

# Mean-centre delivery variables
delivery_means                   = input_data_huc12[delivery_columns].mean()
input_data_huc12[delivery_columns] = input_data_huc12[delivery_columns] - delivery_means


## 6. Build per-year PyTorch DataLoaders

Three `MinMaxScaler`s are fit on **all reach-years at once** (not per-year
means).  HUC12 IDs are derived by stripping the 2-digit year suffix from
`waterid`.


In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
setup_seed(42)

input_data_huc12['HUC12'] = input_data_huc12['waterid'].astype(str).str[:-2]

scaler      = MinMaxScaler().fit(input_data_huc12[input_columns])
scaler_strm = MinMaxScaler().fit(input_data_huc12[input_columns_strm])
scaler_res  = MinMaxScaler().fit(input_data_huc12[input_columns_res])

years = sorted(input_data_huc12['Year'].unique())
yearly_loaders = []

for year in years:
    yd = input_data_huc12[input_data_huc12['Year'] == year].copy()

    X_sc      = torch.tensor(scaler.transform(yd[input_columns].values),
                             dtype=torch.float32, device=device)
    X_sc_strm = torch.tensor(scaler_strm.transform(yd[input_columns_strm].values),
                             dtype=torch.float32, device=device)
    X_sc_res  = torch.tensor(scaler_res.transform(yd[input_columns_res].values),
                             dtype=torch.float32, device=device)
    obs = torch.tensor(yd['depvar'].values, dtype=torch.float32, device=device)
    obs[obs == 0] = float('nan')

    ds = TensorDataset(
        torch.tensor(yd[input_columns].values,    dtype=torch.float32, device=device),
        X_sc, X_sc_strm, X_sc_res,
        torch.tensor(yd[source_columns].values,    dtype=torch.float32, device=device),
        torch.tensor(yd[delivery_columns].values,  dtype=torch.float32, device=device),
        torch.tensor(yd[stream_columns].values,    dtype=torch.float32, device=device),
        torch.tensor(yd[reservoir_columns].values, dtype=torch.float32, device=device),
        torch.tensor(yd['rchtype'].values,   dtype=torch.int64,   device=device),
        torch.tensor(yd['hydseq'].values,    dtype=torch.float32, device=device),
        torch.tensor(yd['headflag'].values,  dtype=torch.int64,   device=device),
        torch.tensor(yd['from_node'].values, dtype=torch.int64,   device=device),
        torch.tensor(yd['to_node'].values,   dtype=torch.int64,   device=device),
        torch.tensor(yd['frac'].values,      dtype=torch.float32, device=device),
        torch.tensor(yd['new_waterid'].values, dtype=torch.int64, device=device),
        torch.tensor(yd['waterid'].values,   dtype=torch.int64,   device=device),
        torch.tensor(yd['iftran'].values,    dtype=torch.int64,   device=device),
        obs,
    )
    yearly_loaders.append(DataLoader(ds, batch_size=len(yd), shuffle=False))

print(f"Created {len(yearly_loaders)} year-loaders ({years[0]}–{years[-1]})")


## 7. Save prepared data

In [ ]:
with open(os.path.join(output_dir, 'prepared_data.pkl'), 'wb') as f:
    pickle.dump({
        'input_data_huc12':   input_data_huc12,
        'dlvdsgn_array':      dlvdsgn_array,
        'delivery_means':     delivery_means,
        'source_columns':     source_columns,
        'delivery_columns':   delivery_columns,
        'stream_columns':     stream_columns,
        'reservoir_columns':  reservoir_columns,
        'input_columns':      input_columns,
        'input_columns_strm': input_columns_strm,
        'input_columns_res':  input_columns_res,
        'scaler':             scaler,
        'scaler_strm':        scaler_strm,
        'scaler_res':         scaler_res,
        'years':              years,
    }, f)
print(f"Saved prepared data → {output_dir}prepared_data.pkl")
